## Gossiping
Avevamo introdotto il gossiping cercando di risolvere il problema della propagazione delle informazioni tra nodi, insieme al flooding e la random walk.  
Con il gossiping in particolare ci si chiede **come diffondere un messaggio senza assegnare al nodo iniziale tutto il lavoro di invio --> l'idea è che anche i nodi che ricevono l'aggiornamento contribuiscano a propagarlo**.

Il **multicast** è un modello di comunicazione dove un messaggio è destinato a un **gruppo di ricevitori**: può essere *uno-a-molti* (es. distribuzione video da sorgente a più utenti) o *molti-a-molti* (es. videoconferenza).  
Il broadcast rappresenta un caso particolare di multicast dove i destinatari sono tutti i nodi della rete.

Per capire il problema di scalabilità, immaginiamo che un nodo $A$ debba mandare lo stesso msg a $B$ e $C$ e che i percorsi verso $B$ e $C$ condividano un link --> se $A$ effettua due invii *unicast* --> su quel link viaggeranno due copie, quando ne bastava una sola...  
Il principio per ridurre la msg complexity è quindi **far viaggiare una sola copia sul link comune e duplicarla alla diramazione**.

Il multicast si può realizzare:
- *a livello di rete* (IP multicast): i router replicano i pacchetti dove i percorsi si diramano, verso le destinazioni appropriate. Tuttavia non tutti i router supportano il multicast..
- **a livello applicativo**: i nodi dell'applicazione sappiamo infatti che scambiano messaggi in una **rete overlay** su cui potremmo realizzare il multicast!

Nell'**overlay strutturata** si possono organizzare percorsi di diffusione espliciti. Le topologie più adatte a questo scopo sono gli *alberi* (dove esiste un unico percorso tra due nodi) e le *mesh* (che invece offrono più percorsi possibili). In particolare è possibile costruire tramite Chord (non lo vediamo) un albero di multicast.

Nell'**overlay non strutturata** invece la propagazione procede tramite i contatti dei nodi, senza un percorso di distribuzione prestabilito per ogni messaggio --> in questo scenario si introduce il gossiping.

---

Similmente alla RW, nel gossiping il nodo **contatta un sottinsieme casuale dei vicini che conosce**, e **i nodi che apprendono l'informazione possono a loro volta diffonderla** --> *il gossiping cerca di ridurre il numero di msg inviati rispetto al flooding, cercando al contempo di essere più veloce rispetto alla RW.*

Il nome richiama "pettegolezzo" non a caso: raccontando un gossip a qualcuno, quello lo potrà raccontare ai propri vicini e così via... si paragona del resto il gossiping anche al concetto di epidemia --> protocolli di gossiping ed epidemici sono della stessa famiglia, ed entrambi non deterministici.    
Un nodo si dice **contaminato** quando ha appreso l'aggiornamento e può contribuire a diffonderlo.

Si collega il gossiping a tre proprietà:
- **Semplicità degli algoritmi**
- **Decentralizzazione**: ogni nodo sceglie i propri contatti casualmente, senza alcuna coordinazione centrale.
- **Scalabilità**: il numero di contatti avviati da ciascun nodo **per ciclo** (la comunicazione in gossip può essere in differita: appena un nodo riceve l'info può decidere di diffonderla dopo un certo tempo) può essere limitato all'aumentare di $N$
- **Robustezza e Resilienza**: l'informazione può raggiungere un nodo attraverso percorsi diversi, per cui se un nodo o link fallisce, l'informazione può comunque arrivare a destinazione (message redundancy).

es. applicativi: Amazon S3 per la diffusione di info utili a evitare nodi irraggiungibili, in Dynamo la failure detection dei nodi, Cassandra per la propagazione di info sul group membership e failure detection....

Due strategie fondamentali per il gossiping: l'**Anti-Entropy** e **Rumor Spreading**.

### Anti-Entropy
L'**Anti-Entropy** parte dallo stato dei nodi e cerca di ridurne le differenze (ridurre in un certo senso cioè il "disordine", da qui il nome!).

Periodicamente, ogni nodo ($P$) sceglie $u.a.r.$ un nodo ($Q$) e ci si scambiano gli aggiornamenti secondo tre modalità, con l'obiettivo di avere lo stesso stato alla fine del processo:
- **Push** $P \Rightarrow Q$: in questo caso se $P$ ha l'aggiornamento, aiuta a diffonderlo mandandolo a $Q$
- **Pull** $P \Leftarrow Q$: in questo caso $P$ chiede gli aggiornamenti, e $Q$ lo aiuta se ha quello che manca a $P$
- **Push-Pull** $P \Leftrightarrow Q$: questa modalità è la più efficiente, perché se uno dei due ha l'aggiornamento, lo trasferisce all'altro.

Il push-pull è il più veloce in quanto all'inizio, quando pochi nodi conoscono la novità, è il *push* a permettere a quei pochi nodi di diffonderla.  
Verso la fine invece, quando quasi tutti la conoscono, il *pull* aiuta gli ultimi nodi in quanto hanno buone probabilità di contattare qualcuno che è già aggiornato.

Nell'immagine seguente, sull'asse x ci sono i **round** (intervalli nei quali ogni nodo avvia uno scambio), mentre sull'asse y i nodi che **ancora non conoscono l'aggiornamento** --> si osserva che la curva che scende più velocemente (e quindi più veloce) è proprio push-pull.

<img src="imgs/7_22.png" width="400">

### Rumor Spreading
Nel **Rumor Spreading** un nodo con un aggiornamento (**contaminato**) sceglie periodicamente $F \ge 1$ (*fanout*) vicini e gli manda l'update per contaminarli a loro volta. Un nodo che contaminato può smettere di diffondere l'aggiornamento con una certa probabilità.

Nell'approccio **feedback-based**, quando un nodo contaminato $P$ contatta un nodo che già conosce l'aggiornamento, smette di propagare quell'aggiornamento con probabilità $p_{stop}$.  
Nell'esempio qui sotto si assume fanout $F = 1$, e si osserva come la frazione dei nodi rimasti incontaminati ($s$, sull'asse $y$) cresce all'aumentare di $p_{stop}$ (sull'asse $x$), arrivando comunque a un $80\%$ di nodi contaminati anche con $p_{stop} = 1$!.

Si può infatti dimostrare che esiste una relazione trascendentale tra $s$ e $p_{stop}$:
$$s = e^{-\left(1+\frac{1}{p_{stop}}\right)(1-s)}$$
che è trascendentale perché l'incognita $s$ compare anche all'esponente.

Per ridurre ancora di più la frazione di nodi incontaminati nel tempo, l'idea è *combinare il rumor spreading con l'anti-entropy*.

---

Il **framework generale** per i protocolli di gossipign è il seguente:

<img src="imgs/7_23.png" width="400">

All'inizio di ogni round, $P$ sceglie chi contattare (`selectPeer`): immaginiamo che scelga $Q$ tra i nodi che conosce --> sceglie cosa inviare (`selcetToSend`, può infatti mandare tutta l'informazione che ha o solo una parte, come gli aggiornamenti recenti) e la invia a $Q$.  
$Q$ riceve la richiesta e, nel paradigma push-pull, sceglie a sua volta cosa inviare a $P$.  
Entrambi decidono quali informazioni mantenere (`selectToKeep`), filtrando cos' eventuali duplicati o scartando info in base alla politica del protocollo.  
Infine, entrambi elaborano la vista aggiornata (`processData(view)`).

I round non richiedono avvio sincronizzato: tutti possono eseguire il protocollo ogni 60 secondi, ma ciascuno può cominciare in momenti diversi.

In questo schema semplice sorgono però diverse questioni:
- **come scegliere i nodi**: diverse alternative, ad es. scegliendo in modo del tutto $u.a.r.$ tra i nodi attualmente attivi, oppure magari scegliendo i peer che sono stati contattati meno e quindi con buona probabilità meno aggiornati (es. CockroachDB)
- **che dati scambiare** e **come processarli**: questo chiaramente dipende specificatamente dall'applicazione

---

Vediamo ora un semplice confronto tra **gossiping e flooding** per capire quali sono i vantaggi e svantaggi del primo rispetto al secondo.  

Poniamoci su una stessa griglia con 9 nodi, tre round, e osserviamo il numero di msg scambiati e quanti nodi vengono raggiunti con nodo iniziatore in alto a sx.  
Nel **flooding** ogni nodo inoltra ad ogni round deterministicamente il msg a tutti i vicini --> dopo tre round 8 su 9 nodi raggiunti e 10 msg scambiati.

<img src="imgs/7_24.png" width="400">

Per il gossiping consideriamo una variante dove un nodo decide di contattare un suo vicino con probabilità $p$ (quindi diverso rispetto all'approccio feedback-based visto prima): in questa specifica esecuzione il numero di nodi raggiunti è 7, ma con solo 7 messaggi inviati!  
Inoltre la caratteristica importante è la **ridondanza**. per cui anche se un nodo non è stato raggiunto in un round precedente ha ancora possibilità di essere informato grazie ad altri suoi vicini.

<img src="imgs/7_25.png" width="400">

Quindi possiamo dire che il **flooding** ha il *vantaggio di coprire tanti nodi in pochi round, ma inondando la rete con tanti messaggi ridondanti* (la msg complexity è infatti lineare rispetto al numero di nodi).  

Il **gossiping** ha invece come caratteristiche chiave il fatto di essere **non deterministica**, fare **decisioni locali che comunque portano a un comportamento globale**, inviare (se ben implementato, nei contesti adatti) **meno messaggi** e avere **ridondanza, quindi robustezza** (se un nodo si guasta, l'informazione può comunque arrivare a destinazione).

Ciò per cui quindi il gossiping vuole migliorare rispetto al flooding è **ridurre il numero di messaggi ridondanti inviati**.  
Gli "*svantaggi*" sono il fatto che, essendo non deterministico, **non è garantito che tutti i nodi vengano raggiunti**, e inoltre mentre il flooding chiede tempo lineare per raggiungere tutti i nodi **con il gossiping tipicamente i tempi sono più lunghi**.

---

Le applicazioni del gossiping sono molteplici:
- **Group membership**: i nodi si comunicano chi conoscono, per far sì che gli altri abbiano una visione più ampia della rete
- **Peer sampling**: ogni nodo costruisce o aggiorna una visione parziale dei peer da cui ricevere/inviare informazioni (es. bootstrapping)
- **Monitoraggio**: si diffondono info sullo stato delle risorse, ad esempio il carico
- **Failure detection**: si raccolgono e propagano "indizi" sul funzionamento dei nodi
- **Calcolo distribuito**: si combinano valori locali per ottenere valori aggregati come media, somma, min, max... (es. per il min ogni nodo conserva il minimo locale e lo aggiorna ad ogni aggiornamento, scegliendo sempre il valore più piccolo tra il proprio e quello ricevuto)

### Gossiping Case Studies
Vediamo tre casi di studio di protocolli di gossiping: **Blind Counter Rumor Mongering**, **Bimodal Multicast** e **Gossip-based Failure Detection**.

#### Blind Counter Rumor Mongering
Questo protocollo si chiama così perché *il rumor mongering indica la diffusione del messaggio che prima o poi si fermerà* (== rumor spreading), *blind* che la regola di arresto è indipendente da un feedback del destinatario (come invece avveniva nel feedback-based rumor spreading, per cui mi fermavo con probabilità $p_{stop}$ quando contattavo un nodo che già conosceva l'aggiornamento) e *counter* ogni nodo usa un contatore per decidere quando fermarsi.

I due parametri del protocollo sono $F$ fanout e $k$ contatore.  
Il nodo iniziale invia $m$ ad $F$ vicini scelti $u.a.r.$ --> per ogni nodo ad ogni round, se il nodo ha ricevuto $\le k$ volte $m$ allora sceglie $F$ vicini $u.a.r.$ **con criterio** a cui inviare $m$, altrimenti non fa nulla. 

**Con criterio** significa che ogni nodo tiene conto dei nodi che gli mandano i messaggi e di quelli a cui hanno inviato messaggi in precedenza --> evitano così di mandare messaggi ridondanti a nodi già informati.

Con $F = k = 2$ si sono osservati sperimentalmente i seguenti risultati rispetto al flooding:
- circa $50%$ di messaggi risparmiati
- copertura del $90%$ dei nodi (che comunque, sapendo di aver dimezzato la msg complexity, è parecchio)
- diffusione però circa due volte più lenta rispetto al flooding (come ci aspettavamo, avendo parlato dei limiti del gossiping poco fa)

#### Bimodal Multicast
Il Bimodal Multicast, chiamato anche **pbcast**, *si pone uno scenario dove possono esservi perdite di messaggi* --> vuole offrire un *multicast affidabile* in questo contesto.  
Il protocollo cerca di distribuire rapidamente un msg e poi di recuperare eventuali consegne mancanti.

Due attività:
1. **Message Distribution**: l'invio iniziale in multicast, senza garanzie di recupero dei messaggi persi
2. **Gossip Repair**: periodicamente i nodi si scambiano informazioni tramite gossip sui messaggi ricevuti e recuperano eventuali messaggi persi.

Un esempio applicativo di questo protocollo è **Fastly**, che usa un approccio di questo tipo per l'invalidazione della cache: se un contenuto cambia, i nodi della CDN devono sapere che la copia conservata è ormai obsoleta.

Nell'esempio, la prima fase inizia quando due nodi $p_1$ e $p_2$ tentano il multicast (senza garanzie di consegna) verso tutti gli altri nodi. Le frecce rosse indicano i messaggi persi.  

<img src="imgs/7_26.png" width="400">

La seconda fase prevede quindi il repair: periodicamente (es. ogni 100 ms) ogni nodo manda un **digest** del proprio stato a nodi scelti $u.a.r.$ (digest = informazione che indica quali messaggi possiede, senza contenere i messaggi stessi).  
Il destinatario confronta il digest con la propria storia. Se vede che l'altro possiede un msg che gli manca --> **ne richiede una copia**, e allora il mittente originale gli manda il msg richiesto.

<img src="imgs/7_27.png" width="400">

**Perché si chiama bimodale?** La motivazione non riguarda il fatto che agisce su due fasi, ma **la distribuzione dei risultati di consegna**: nel modello analizzato, gli esiti si concentrano vicino a due valori estremi:
- o *il msg raggiunge pochissimi nodi* (ad es. quando il mittente originale fallisce moolto presto --> pochi nodi informati, la fase di recupero non è sufficiente..)
- *il msg raggiunge quasi tutti i nodi* grazie alla diffusione sufficiente e quindi fase di recupero più efficace.

Gli esiti medi, al contrario, hanno probabilità bassissima --> garanzia probabilistica del tipo "o a quasi tutti o a pochissimi", *atomicity*.

<img src="imgs/7_28.png" width="400">

#### Gossip-based Failure Detection
Un **failure detector** cerca di individuare quali nodi **si sospetta abbiano subito un crash**. Il termine "sospetta" è fondamentale, in quanto l'assenza di notizie può anche dipendere da un nodo lento, sovraccarico o da un problema di rete.

Nei sistemi su larga scala, i requisiti fondamentali per il problema di failure detection sono:
- *Completeness*: prima o poi, si devono identificare i nodi falliti
- *Accuracy*: minimizzare i Falsi Positivi (sospettare nodi attivi come falliti)
- *Speed of Detection*
- *Scalability*: i messaggi scambiati devono essere sub-lineari rispetto al numero di nodi

Un modo tipico di fare failure detection è tramite **heartbeat**, e ci sono diversi approcci: *centralizzato* se tutti mandano heartbeat a un singolo monitor centrale (ma questo problematico perché rappresenta SPOF), *all-to-all* oer cui ogni nodo invia heartbeat a tutti gli altri (qui problematico perché $N(N-1)$ messaggi scambiati periodicamente... non scalabile) e **Gossip-based** per cui invece *ogni nodo contatta pochi peer e propaga informazioni di vitalità*, è quello su cui ci focalizziamo.

Ci concentriamo su un protocollo implementato come segue: ogni nodo $i$ conserva una tabella. Per ciascun nodo del sistema $j$, il nodo $i$ registra:
- $H_j$ == l'ultimo heartbeat ricevuto da $j$ (espresso come contatore incrementale)
- $T_j$ == l'ora locale in cui ha osservato l'ultimo aumento di quel contatore

Il contatore di $j$ **viene incrementato da** $j$: gli altri nodi ne diffondono semplicemente il valore.  
Ogni periodo $T_{gossip}$, il nodo incrementa il proprio contatore ed invia la tabella a un peer scelto $u.a.r.$. Chi riceve confronta ogni voce con la propria:
- **se il contatore ricevuto è maggiore** --> conserva il nuovo valore e imposta $T_j$ all'ora locale corrente
- **se il contatore ricevuto è uguale o minore** --> mantiene la voce precedente, incluso il tempo

Per la failure detection, definiamo $T_{fail}$: rappresenta **il tempo massimo che un nodo aspetta senza osservare un aumento del contatore di heartbeat di un altro nodo, prima di sospettarne il guasto**.  
Ad es. supponiamo che $P$ tenga in osservazione un nodo $j$ e ha memorizzato $H_j = 9$ e $T_j = 45$ (ora locale), significa cioè che al proprio tempo locale 45 ha osservato l'ultimo heartbeat di $j$ con contatore 9.  
Allora, con $T_{fail} = 10$, se al tempo locale 56 non ha ancora osservato un aumento del contatore di $j$ --> sospetta che $j$ sia fallito.

Nell'esempio qui sotto, il nodo 2 riceve la tabella del nodo 4 al proprio tempo locale 70, con $T_{fail} = 20$. Aggiorna solo i contatori più alti rispetto ai suoi, e in particolare non aggiorna il nodo 3. Questo era stato aggiornato all'ora locale 45, ma l'ora locale attuale del nodo 2 è 70 --> $70 - 45 = 25 > T_{fail} = 20$ --> sospetta che sia fallito...

<img src="imgs/7_29.png" width="400">

I parametri in questo protocollo regolano il tradeoff tra **velocità**, **traffico** e **falsi positivi**:
- $T_{gossip}$: più è piccolo, più velocemente si diffondono le informazioni, ma più messaggi vengono inviati
- $T_{fail}$: più è piccolo, più velocemente si sospetta un nodo fallito, ma più falsi positivi si generano (es. se un nodo è lento o la rete è congestionata)

Per evitare i falsi positivi, un valore $T_{cleanup}$ (tipicamente dello stesso ordine di $T_{fail}$) fa si che i nodi non siano eliminati prima della sua scadenza anche se sospetti.  

Nel protocollo semplificato visto ora, un altro problema riguarda la **dimensione dei messaggi**, qui una tabella con $O(N)$ voci --> per ridurre il traffico, diverse ottimizzazioni (come inviare solo la parte della tabella con aggiornamenti recenti)... rappresenta un'applicazione concreta del `selectToSend` visto nel framework iniziale!


Si citano tre sistemi che sfruttano Gossip per il failure detection:
- **SWIM** (usato da Serf e Consul) controlla un nodo casuale con $ping$: se manca la risposta, chiede ad altri $k$ nodi di contattarlo indirettamente. Se ancora no `ack` --> il nodo viene marcato come *sospetto*, e si dichiara fallito solo dopo un timeout.
- **Cassandra** combina il gossip con un $\phi$-accrual failure detector: intuitivamente, si usa la storia degli intervalli di arrivo degli heartbeat per stimare la probabilità che un nodo sia fallito, senza limitarsi a un timeout fisso.
- **Redis Cluster** usa gossip per propagare informazioni sullo stato dei nodi e dei cluster, e per rilevare nodi falliti o non raggiungibili.

---

Il Gossiping è usato anche in applicazioni piuttosto recenti
- Nel **ML distribuito** i nodi possono calcolare localmente i gradienti e poi diffonderli tramite gossip, riducendo il carico di comunicazione rispetto a un approccio centralizzato.
- Nella **swarm robotics** vedi il signor Marco Rodigo: i robots usano il gossip per condividere informazioni in modo del tutto decentralizzato e svolgere task anche complessi
- **Decentralized RAG (Retrieval-Augmented Generation)**: il RAG è un metodo usato per ridurre l'allucinazione dei modelli di linguaggio --> l'applicazione del gossip può contribuire allo scambio di informazioni tra nodi che hanno accesso a diverse fonti di conoscenza, migliorando la qualità delle risposte generate